## Taller "Diseño e implementación de flujos de trabajo agénticos"
### I Congreso de Inteligencia Artificial, UNAH
### Tegucigalpa, Honduras, 13-17 julio de 2026 

## Endpoint de LLM 

Para la implementación de los flujos de trabajo y los ejercicios, vamos a consumir un endpoint gratuito de un modelo de lenguaje. Ofrecemos dos alternativas populares: Gemini y Groq.




**Acceso a Gemini**

-Dirigirse a la página https://aistudio.google.com

-Buscar opción de Crear clave de API. 

-Dar nombre a proyecto de Gemini y crear clave. 

-Copiar clave e insertar en el código. 

**Acceso a Groq**

-Dirigirse a la página https://console.groq.com/home

-Registrarse (basta una cuenta de Google).

-Presionar "Create API Key".

-Copiar clave e insertar en el código.

Si bien los modelos de Gemini son excelentes, su uso gratuito es limitado. Aquí usaremos un endpoint de LLM proporcionado por Groq, que nos dará mayor libertad para experimentar. 

### Instalación de dependencias 

In [ ]:
!pip install langchain_openai -q

In [ ]:
!pip install datasets -q

### Preámbulo y llamada de LLM 

Para consumir el endpoint de Gemini, usaremos la función ChatOpenAI, que es compatible con LangChain.

In [ ]:
#Importaciones 
from langchain_core.prompts import PromptTemplate 
from langchain_openai import ChatOpenAI


In [ ]:
#Gemini
API_KEY = 

In [ ]:
#Groq 
API_KEY = 

In [ ]:
#Llamada a modelo Gemini
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    model="gemini-3.5-flash",
    temperature=0.9
)


In [ ]:
#Llamada a modelo mediante Groq
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model="llama-3.3-70b-versatile", # or "llama-3.1-8b-instant" for speed
    temperature=0.9
)


## Chain prompting

En el consumo de modelos de lenguaje (LLM), el diseño de prompts (prompt engineering) es una destreza esencial. Para tareas sencillas, un prompt suele bastar. Sin embargo, la solución a problemas prácticos suele requerir de la ejecución de tareas más complejas. El encadenamiento de prompts (chain prompting) es una estrategia que consiste en dividir la tarea principal en partes más sencillas que son ejecutadas por llamadas separadas a un LLM.





**Ventajas**

-Mayor precisión en la ejecución de tareas complejas. 

-Cada etapa puede ser probada independientemente. 


**Desventajas**

-Consumo de múltiples llamadas de LLM.

-Tiempo de latencia mayor.

**Diagrama de flujo**

### Ejemplos 

_Ejemplo 1: Creando historias._ Propondremos a nuestro modelo la siguiente tarea.

-Escribir una historia basada en un tema propuesto por el usuario.

-Resumir la historia generada en el paso anterior. 

In [ ]:
#Plantillas de prompt
story_prompt = PromptTemplate(
    input_variables=["genre"],
    template="Escribe una historia de {genre} en 3 o 4 párrafos máximo."
)

summary_prompt = PromptTemplate(
    input_variables=["story"],
    template="Haz un resumen de la siguiente historia en una sola oración:\n{story}"
)


In [ ]:
#Cadena de prompts 
def story_chain(genre):
    story = (story_prompt | llm).invoke({"genre": genre}).content
    summary = (summary_prompt | llm).invoke({"story": story}).content
    return story, summary


In [ ]:
genre = "misterio"
story, summary = story_chain(genre)

In [ ]:
print(story)

In [ ]:
print(summary)

Podemos comprobar que en este caso es posible diseñar un prompt que ejecute ambas tareas al mismo tiempo. 

In [ ]:
cot_prompt=PromptTemplate(
    input_variables=["genre"],
    template="Tienes dos tareas, Primero, escribe una historia de {genre} en 3 o 4 párrafos máximo. Luego debes resumir la historia en una sola oración."
)

In [ ]:
genre='misterio'
new_history = (cot_prompt | llm).invoke({"genre": genre}).content

In [ ]:
new_history

*Ejemplo 2: Diseñando un cuestionario.* En este ejemplo, los prompts encadenados se irán generando de manera dinámica. Nos interesa diseñar un cuestionario alreadedor de una pregunta inicial proporcionada por el usuario. 

In [ ]:
#Prompt de pregunta inicial
answer_prompt = PromptTemplate(
    input_variables=["question"],
    template="Responde de manera concisa la siguiente pregunta:\n{question}"
)


In [ ]:
follow_up_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="Basándose en la pregunta '{question}' y en la respuesta '{answer}', genera una pregunta de seguimiento relevante. Devuelve solo el texto de la pregunta."
)


In [ ]:
def dynamic_qa(initial_question, num_follow_ups=2):
    qa_chain = []
    current_question = initial_question

    for _ in range(num_follow_ups + 1):  
        answer = (answer_prompt | llm).invoke({"question": current_question}).content
        qa_chain.append({"question": current_question, "answer": answer})
        
        if _ < num_follow_ups: 
            current_question = (follow_up_prompt | llm).invoke({"question": current_question, "answer": answer}).content

    return qa_chain


In [ ]:
# Test the dynamic Q&A system
initial_question = "Cuál es el producto más famoso de Honduras?"
qa_session = dynamic_qa(initial_question)

for i, qa in enumerate(qa_session):
    print(f"Q{i+1}: {qa['question']}")
    print(f"A{i+1}: {qa['answer']}\n")

## Routing 

En la solución de algunos problemas, la estrategia natural consiste en dividir en varios casos independientes. En la arquitectura router, recurrimos a un modelo de lenguaje para decidir de manera autónoma el caso apropiado a seguir dentro del flujo.

**Ventajas**

-Decisiones autónomas sin intervención de frontend y backend.

-Flexibilidad para abordar diversidad de solicitudes. 



**Desventajas**

-Corregir errores de clasificación puede ser complicado. 

-La capa adicional de routing añade latencia y costo. 

### Dataset Nemotron personas 

Para algunos ejercicios, utilizaremos un dataset sintético (Nemotron Personas El Salvador), que está disponible de manera gratuita en la siguiente dirección.

https://huggingface.co/datasets/nvidia/Nemotron-Personas-El-Salvador

Nos interesa este dataset por la variedad de registros (148,000 personas) y campos (24), por lo que constituye un excelente ejemplo de uso en análisis de datos. Por otra parte, notar que no hay una división entre entrenamiento y validación. 

**Pasos para uso de dataset**

-Importar librería datasets.

-Descargar repositorio. 

-Convertir a Pandas para fácil consumo. 

In [ ]:
from datasets import load_dataset
import pandas as pd


In [ ]:
#Carga de dataset del repositorio HuggingFace
dataset = load_dataset("nvidia/Nemotron-Personas-El-Salvador")


In [ ]:
#Conversión a dataframe de Pandas 
df = dataset['train'].to_pandas()

print(f"DataFrame shape: {df.shape}")

In [ ]:
df.head()

### Ejemplos

*Ejemplo 1: Menú de un restaurante.* Un restaurante desea utilizar un servicio de chatbot, que responda a solicitudes de clientes. El menú consta de tres categorías: comidas, bebidas y postres. Deseamos diseñar un flujo que clasifique las consultas del usuario en una de estas tres categorías. 

In [ ]:
#Llamada a modelo mediante Groq
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model="llama-3.3-70b-versatile", 
    temperature=0.1 #Temperatura baja para clasificador
)

In [ ]:
#Plantilla de prompt
prompt_template = PromptTemplate.from_template(
    """Eres un asistente en un restaurante. Tu tarea es clasificar la siguiente consulta del cliente en exactamente una de estas tres categorías: 'comidas', 'bebidas' o 'postres'.

REGLA ESTRICTA: Responde ÚNICAMENTE con la palabra de la categoría en minúsculas. Sin puntos, sin comillas, sin explicaciones y sin texto adicional.

Consulta: {query}
Categoría:"""
)


In [ ]:
cadena_clasificacion = prompt_template | llm


In [ ]:

# 6. Función que ejecuta la clasificación
def clasificar_pedido(query):
    try:
        # Ejecutamos la cadena pasando un diccionario con la variable 'query'
        respuesta_llm = cadena_clasificacion.invoke({"query": query})
        
        # Limpiar la respuesta (quitar espacios, saltos de línea, puntos o comillas)
        categoria_elegida = respuesta_llm.content.strip().lower().replace(".", "").replace("\"", "").replace("'", "")
        
        # Validación de seguridad (Fallback)
        if categoria_elegida not in ["comidas", "bebidas", "postres"]:
            categoria_elegida = f"DESCONOCIDO ({categoria_elegida})"
            
        print(f"Resultado de la clasificación: [{categoria_elegida.upper()}]\n")
        
    except Exception as e:
        print(f"Error al clasificar: {e}\n")


In [ ]:
query='Quiero una orden de papas'

In [ ]:
clasificar_pedido(query)

*Ejemplo 2: Router por columnas.* Usando el dataset sugerido (Nemotron Personas El Salvador), deseamos clasificar descripciones de personas en algunos de los campos del dataset, representados por columnas.

In [ ]:
PERSONA_COLUMNS = [
    "professional_persona",
    "sports_persona",
    "arts_persona",
    "travel_persona",
    "culinary_persona",
    "family_persona",
    "persona",
]


In [ ]:
def route_query(query: str) -> str:
    prompt = f"""
Tu tarea es relacionar una consulta del usuario con las columnas de una dataframe.

Debes seleccionar UNA ÚNICA columna donde se debe realizar la búsqueda semántica.

Consulta:
{query}

Responde únicamente con el nombre exacto de una de las siguientes columnas:

professional_persona
sports_persona
arts_persona
travel_persona
culinary_persona
family_persona
persona

No escribas explicaciones ni texto adicional.
"""
    
    response = llm.invoke(prompt)

    selected_column = response.content.strip()

    if selected_column not in PERSONA_COLUMNS:
        selected_column = "persona"

    return selected_column




In [ ]:
query='tengo dos hermanos'

In [ ]:
route_query(query)